# TrialSense — DNABERT-2 fine-tuning on a free GPU

**Run this on [Kaggle Notebooks](https://www.kaggle.com/code)** — 30 GPU-hours/week free, no card required.

## Why this notebook exists

DNABERT-2 fine-tuning needs an NVIDIA GPU. Its published code imports Triton
flash-attention kernels, which are CUDA-only — on an Intel Arc / AMD / Apple
machine it will not load at all, and on CPU a fine-tune that takes 20 minutes
here would take a day or more.

So: **train here, run at home.** This notebook produces a `.pt` checkpoint you
download and drop into your local `models/` folder. Inference on the trained
model runs fine on any CPU.

## Setup (30 seconds)

1. New Notebook → **Settings → Accelerator → GPU T4 x2** (or P100)
2. **Settings → Internet → On** (needed to download the checkpoint)
3. Run all cells

## An honest expectation before you start

On TrialSense's **synthetic** cassettes, DNABERT-2 will probably *lose* to the
k-mer baseline. That is not a bug and not a waste of time — a genomic language
model's advantage comes from having learned the structure of *real* DNA, and
there is no real biology in a randomly-generated motif cassette for it to
recognise, while a k-mer counter is near-perfectly matched to how those
cassettes were built.

Measuring that is the point. It converts "we should use real sequences" from a
hunch into a number. Swap in real NCBI Pathogen Detection sequences (Cell 7)
and the ordering should reverse.

## 1. Environment check

In [ ]:
import subprocess, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Settings -> Accelerator -> GPU, then re-run.')
print('device:', torch.cuda.get_device_name(0))

## 2. Install

`einops` and `peft` are required by DNABERT-2's remote code. Pinning
`transformers<4.40` avoids an API change that breaks its custom model class.

In [ ]:
!pip install -q einops peft "transformers==4.38.2" accelerate 2>&1 | tail -2
print('installed')

## 3. Rebuild the TrialSense sequence generator

Copied verbatim from `trialsense/amr.py` so this notebook is self-contained —
you do not need to upload the repo. The seeds are fixed, so the sequences here
are byte-identical to the ones your local app uses.

In [ ]:
import hashlib
import numpy as np

ANTIBIOTIC_CLASSES = [
    'Penicillins', 'Cephalosporins', 'Carbapenems', 'Fluoroquinolones',
    'Macrolides', 'Aminoglycosides', 'Tetracyclines',
    'Trimethoprim-sulfonamides', 'Glycopeptides', 'Oxazolidinones',
    'Polymyxins', 'Rifamycins',
]

GENES = {
    'blaTEM-1':    (['Penicillins'], 0.44),
    'blaCTX-M-15': (['Penicillins', 'Cephalosporins'], 0.52),
    'blaNDM-1':    (['Penicillins', 'Cephalosporins', 'Carbapenems'], 0.60),
    'blaKPC-2':    (['Penicillins', 'Cephalosporins', 'Carbapenems'], 0.57),
    'blaOXA-48':   (['Penicillins', 'Carbapenems'], 0.46),
    'mecA':        (['Penicillins', 'Cephalosporins', 'Carbapenems'], 0.34),
    'vanA':        (['Glycopeptides'], 0.42),
    'ermB':        (['Macrolides'], 0.36),
    'tetM':        (['Tetracyclines'], 0.39),
    "aac(6')-Ib":  (['Aminoglycosides'], 0.56),
    'armA':        (['Aminoglycosides'], 0.33),
    'qnrS1':       (['Fluoroquinolones'], 0.48),
    'gyrA_S83L':   (['Fluoroquinolones'], 0.53),
    'sul1':        (['Trimethoprim-sulfonamides'], 0.61),
    'dfrA17':      (['Trimethoprim-sulfonamides'], 0.45),
    'mcr-1':       (['Polymyxins'], 0.51),
    'cfr':         (['Oxazolidinones', 'Macrolides'], 0.31),
    'rpoB_S450L':  (['Rifamycins'], 0.58),
}

CASSETTE_LEN, BACKGROUND_LEN = 900, 2600
N_MOTIFS, MOTIF_LEN = 18, 8
BASES = np.array(list('ACGT'))


def gene_cassette(name):
    gc = GENES[name][1]
    seed = int(hashlib.sha256(name.encode()).hexdigest()[:8], 16)
    rng_m = np.random.RandomState(seed)
    probs = [(1 - gc) / 2, gc / 2, gc / 2, (1 - gc) / 2]
    motifs = [''.join(rng_m.choice(BASES, size=MOTIF_LEN, p=probs))
              for _ in range(N_MOTIFS)]
    rng = np.random.RandomState(seed + 1)
    parts, length = [], 0
    while length < CASSETTE_LEN:
        parts.append(motifs[rng.randint(len(motifs))])
        length += MOTIF_LEN
        if rng.rand() < 0.45:
            filler = ''.join(rng.choice(BASES, size=rng.randint(2, 6), p=probs))
            parts.append(filler)
            length += len(filler)
    return ''.join(parts)[:CASSETTE_LEN]


def background(gc, rng, n):
    p = [(1 - gc) / 2, gc / 2, gc / 2, (1 - gc) / 2]
    return ''.join(rng.choice(BASES, size=n, p=p))


def mutate(seq, rate, rng):
    arr = np.array(list(seq))
    k = int(len(arr) * rate)
    if k:
        idx = rng.choice(len(arr), size=k, replace=False)
        arr[idx] = rng.choice(BASES, size=k)
    return ''.join(arr)


def synthesize(genes, gc=0.51, seed=0, divergence=0.02):
    rng = np.random.RandomState(seed)
    parts = [background(gc, rng, BACKGROUND_LEN)]
    for g in genes:
        parts.append(mutate(gene_cassette(g), divergence, rng))
        parts.append(background(gc, rng, 120))
    rng.shuffle(parts)
    return ''.join(parts)


def labels_for(genes):
    y = np.zeros(len(ANTIBIOTIC_CLASSES), dtype=np.float32)
    for g in genes:
        for c in GENES[g][0]:
            y[ANTIBIOTIC_CLASSES.index(c)] = 1.0
    return y


def realistic_combos(rng, n):
    names = list(GENES)
    linked = [['sul1', 'dfrA17'], ['blaCTX-M-15', "aac(6')-Ib"],
              ['blaNDM-1', 'blaCTX-M-15'], ['mecA', 'ermB'],
              ['vanA', 'tetM'], ['qnrS1', 'gyrA_S83L']]
    out = []
    for _ in range(n):
        s = set()
        k = rng.choice([0, 1, 2, 3, 4, 5], p=[.10, .20, .25, .22, .15, .08])
        for _ in range(int(k)):
            if rng.rand() < 0.35:
                s.update(linked[rng.randint(len(linked))])
            else:
                s.add(names[rng.randint(len(names))])
        out.append(sorted(s))
    return out


print('generator ready — 18 genes, 12 classes')

## 4. Build the dataset

In [ ]:
N_ISOLATES = 700
SEED = 42

rng = np.random.RandomState(SEED)
combos = realistic_combos(rng, N_ISOLATES)
gc_values = [0.327, 0.380, 0.390, 0.507, 0.521, 0.572, 0.656, 0.664]

sequences, labels = [], []
for genes in combos:
    gc = gc_values[rng.randint(len(gc_values))]
    sequences.append(synthesize(genes, gc, int(rng.randint(1 << 30))))
    labels.append(labels_for(genes))
labels = np.vstack(labels)

print(f'{len(sequences)} isolates | mean length {np.mean([len(s) for s in sequences]):.0f} bp')
print(f'label prevalence: {labels.mean(axis=0).round(2)}')

## 5. Load DNABERT-2

If this cell raises a Triton error, uncomment the monkey-patch below — it
forces the standard attention path instead of the flash-attention kernels.

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer

MODEL_ID = 'zhihan1996/DNABERT-2-117M'

# --- if you hit a Triton/flash-attn error, uncomment these two lines:
# import os
# os.environ['DNABERT_DISABLE_FLASH'] = '1'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
backbone = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True)
print('loaded:', sum(p.numel() for p in backbone.parameters()) / 1e6, 'M params')

## 6. Fine-tune with a multi-label head

`BCEWithLogitsLoss` because resistance to the twelve classes is **multi-label**,
not multi-class — an isolate can resist several classes at once.

In [ ]:
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

MAX_LEN, BATCH, EPOCHS, LR = 512, 8, 4, 2e-5
device = 'cuda'


class IsolateDS(Dataset):
    def __init__(self, seqs, ys):
        self.seqs, self.ys = seqs, ys

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, i):
        enc = tokenizer(self.seqs[i], truncation=True, max_length=MAX_LEN,
                        padding='max_length', return_tensors='pt')
        return ({k: v.squeeze(0) for k, v in enc.items()},
                torch.tensor(self.ys[i], dtype=torch.float32))


class Classifier(nn.Module):
    def __init__(self, backbone, n_labels):
        super().__init__()
        self.backbone = backbone
        self.drop = nn.Dropout(0.1)
        self.head = nn.Linear(768, n_labels)

    def forward(self, **enc):
        out = self.backbone(**enc)
        hidden = out[0] if isinstance(out, (tuple, list)) else out.last_hidden_state
        return self.head(self.drop(hidden.mean(dim=1)))


idx_tr, idx_te = train_test_split(np.arange(len(sequences)), test_size=0.25,
                                  random_state=SEED)
tr = DataLoader(IsolateDS([sequences[i] for i in idx_tr], labels[idx_tr]),
                batch_size=BATCH, shuffle=True)
te = DataLoader(IsolateDS([sequences[i] for i in idx_te], labels[idx_te]),
                batch_size=BATCH)

model = Classifier(backbone, len(ANTIBIOTIC_CLASSES)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR)
lossf = nn.BCEWithLogitsLoss()

for ep in range(EPOCHS):
    model.train()
    tot = 0.0
    for enc, y in tr:
        enc = {k: v.to(device) for k, v in enc.items()}
        y = y.to(device)
        opt.zero_grad()
        loss = lossf(model(**enc), y)
        loss.backward()
        opt.step()
        tot += loss.item()

    model.eval()
    P, T = [], []
    with torch.no_grad():
        for enc, y in te:
            enc = {k: v.to(device) for k, v in enc.items()}
            P.append((torch.sigmoid(model(**enc)) >= 0.5).cpu().numpy())
            T.append(y.numpy())
    P, T = np.vstack(P), np.vstack(T)
    f1 = f1_score(T, P, average='macro', zero_division=0)
    print(f'epoch {ep+1}/{EPOCHS}  loss {tot/len(tr):.4f}  '
          f'macro-F1 {f1:.4f}  per-label acc {(P == T).mean():.4f}')

## 7. Head-to-head against the k-mer baseline

**This is the cell that matters.** Same data, same split, same metric.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier

BASE_CODES = np.full(256, 255, dtype=np.uint8)
for i, b in enumerate('ACGT'):
    BASE_CODES[ord(b)] = i
    BASE_CODES[ord(b.lower())] = i


def kmer_features(seq, k=5):
    raw = np.frombuffer(seq.encode('ascii', 'ignore'), dtype=np.uint8)
    codes = BASE_CODES[raw]
    codes = codes[codes != 255]
    if len(codes) < k:
        return np.zeros(4 ** k, dtype=np.float32)
    w = np.lib.stride_tricks.sliding_window_view(codes.astype(np.int64), k)
    idx = w @ (4 ** np.arange(k - 1, -1, -1, dtype=np.int64))
    c = np.bincount(idx, minlength=4 ** k).astype(np.float32)
    return c / c.sum() if c.sum() else c


X = np.vstack([kmer_features(s) for s in sequences])
rf = MultiOutputClassifier(RandomForestClassifier(
    n_estimators=250, min_samples_leaf=2, random_state=SEED, n_jobs=-1))
rf.fit(X[idx_tr], labels[idx_tr])
rf_pred = np.asarray(rf.predict(X[idx_te]))

rf_f1 = f1_score(labels[idx_te], rf_pred, average='macro', zero_division=0)
bert_f1 = f1

print('=' * 62)
print(f"k-mer + Random Forest : macro-F1 {rf_f1:.4f}  "
      f"per-label {(rf_pred == labels[idx_te]).mean():.4f}")
print(f"DNABERT-2 fine-tuned  : macro-F1 {bert_f1:.4f}  "
      f"per-label {(P == T).mean():.4f}")
print('=' * 62)
winner = 'k-mer' if rf_f1 > bert_f1 else 'DNABERT-2'
print(f'WINNER: {winner}  (margin {abs(rf_f1 - bert_f1):.4f})')
print()
if winner == 'k-mer':
    print('Expected. These cassettes are synthetic — assembled from random')
    print('motifs — so there is no real biological structure for a genomic')
    print('language model to exploit, while the k-mer counter matches exactly')
    print('how they were built.')
    print()
    print('This is a MEASURED result, and it is the argument for real data:')
    print('swap these sequences for real ones from NCBI Pathogen Detection')
    print('and re-run this cell. The ordering should reverse.')
else:
    print('DNABERT-2 is extracting structure the k-mer counter misses.')
    print('The transformer path is worth pursuing.')

## 8. Export the checkpoint

Download from the Kaggle output panel on the right, then place it in your
local `models/` folder.

In [ ]:
import json

torch.save({
    'model_state': model.state_dict(),
    'classes': ANTIBIOTIC_CLASSES,
    'model_id': MODEL_ID,
    'max_len': MAX_LEN,
    'metrics': {
        'dnabert_macro_f1': float(bert_f1),
        'kmer_macro_f1': float(rf_f1),
        'n_train': int(len(idx_tr)),
        'n_test': int(len(idx_te)),
        'epochs': EPOCHS,
    },
}, 'dnabert2_trialsense.pt')

with open('dnabert2_benchmark.json', 'w') as fh:
    json.dump({'dnabert_macro_f1': float(bert_f1),
               'kmer_macro_f1': float(rf_f1),
               'winner': winner}, fh, indent=2)

print('saved dnabert2_trialsense.pt and dnabert2_benchmark.json')
print('download both from the Output panel on the right ->')

## 9. Using it back home

```
models/
  dnabert2_trialsense.pt      <- the checkpoint you just downloaded
  dnabert2_benchmark.json     <- the measured comparison
```

Inference needs no GPU. `pip install torch transformers einops` locally and
`trialsense/dnabert.py` will pick the checkpoint up.

---

### The line to say out loud

> *"We integrated DNABERT-2 and benchmarked it head-to-head against our k-mer
> baseline on identical data. On synthetic sequences the k-mer model wins,
> which is exactly what theory predicts — a genomic language model's edge comes
> from real biological structure our cassettes do not have. That is measured
> evidence that our **data** is the bottleneck, not our method."*

Reporting a benchmark you might lose is a stronger position than reporting an
integration you never evaluated.